[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/primers/primer-2-spacy.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Primer 2 — spaCy

**MSIS · AME 5003 and AME 5053 · Practice notebook · about 1 hour · not assessed**

*One pipeline, one call: tokens, lemmas, tags, entities and the dependency parse, and the
attribute names that carry them.*

Primer 1 assembled a pipeline by hand — tokenise, tag, map the tag, lemmatise. spaCy takes the
opposite position. You load one object, call it on a string, and everything has already been
computed; you read the results off the tokens as attributes.

That is the whole design, and it is worth stating plainly because it explains every API decision
you are about to meet. spaCy is what the labs use whenever a task needs more than counting, and
it is what session 6 and session 7 demonstrate from.

This notebook is practice, not assessment. Nothing here is marked.

**By the end you will be able to:**

1. Load a pipeline and run it on text
2. Read what the pipeline decided from `Token` and `Span` attributes
3. Find named entities, with their labels and their positions in the string
4. Use the dependency parse to pull out noun phrases and a verb's subject
5. Write a rule-based `Matcher` pattern, and process a lot of documents efficiently

---
## Part 0 — Setup

Two things to install: the library, and a **model**. They are separate downloads because the
library is code and the model is trained weights — about 12 MB of them for the small English
pipeline. pip installs the first and not the second, which is why a fresh machine that has spaCy
still raises `OSError: Can't find model 'en_core_web_sm'`.

In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm -q
print("Done.")

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

print(nlp.meta["name"], nlp.meta["version"])
print("pipeline:", nlp.pipe_names)
# Verified output:
#   core_web_sm 3.8.0
#   pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

> **Save your own copy now:** File → Save a copy in Drive.

`nlp.pipe_names` lists the components that will run, in order, every time you call `nlp(text)`.
The tokeniser is not in that list because it is not optional — it always runs first, turning the
string into tokens, and the components then annotate those tokens in turn.

The names are worth knowing. `tok2vec` runs first and turns each token into a vector that
depends on its neighbours; the components after it read those vectors rather than the raw words.
`tagger` assigns parts of speech, `parser` builds the dependency tree and finds sentence
boundaries, `ner` finds named entities, `lemmatizer` produces lemmas using the tagger's output,
and `attribute_ruler` fixes up a few tag mappings between them.

That first component is worth remembering when we reach session 32 and contextual embeddings:
the idea that a token's representation should depend on its context is already inside this
pipeline, several years before the models that made it famous.

There are larger English models — `en_core_web_md` and `en_core_web_lg`, which are more accurate
and carry real word vectors — but `sm` is the one this course uses, because it downloads in
seconds and the difference does not change any conclusion we draw.

---
## Part 1 — Doc, Token, and the trailing underscore

Calling `nlp(text)` runs the whole pipeline and returns a `Doc`. A `Doc` behaves like a list of
`Token` objects, and each token carries everything the pipeline decided about it.

In [ ]:
doc = nlp("Dr. Rao didn't book the 9:30 flight to Bengaluru.")

print(f"{'text':10} {'lemma':10} {'pos_':7} {'tag_':6} {'is_stop':8} is_punct")
for token in doc:
    print(f"{token.text:10} {token.lemma_:10} {token.pos_:7} {token.tag_:6} "
          f"{str(token.is_stop):8} {token.is_punct}")
# Verified output:
#   text       lemma      pos_    tag_   is_stop  is_punct
#   Dr.        Dr.        PROPN   NNP    False    False
#   Rao        Rao        PROPN   NNP    False    False
#   did        do         AUX     VBD    True     False
#   n't        not        PART    RB     True     False
#   book       book       VERB    VB     False    False
#   the        the        DET     DT     True     False
#   9:30       9:30       NUM     CD     False    False
#   flight     flight     NOUN    NN     False    False
#   to         to         ADP     IN     True     False
#   Bengaluru  Bengaluru  PROPN   NNP    False    False
#   .          .          PUNCT   .      False    True

Every column in that table came from one call. Compare it with primer 1, where the same
information took a tokeniser, a tagger, a tag-mapping dictionary and a lemmatiser wired together
by hand.

Read the columns as pairs. `pos_` is the coarse class — `NOUN`, `VERB`, `PROPN` — and `tag_` is
the fine-grained Penn tag underneath it. `is_stop` is spaCy's own stop list, so you get stop-word
removal without importing anything. And `didn't` was split into `did` and `n't`, with `n't`
lemmatised to `not`, which is exactly what a sentiment task needs.

### The trailing underscore

This trips up everyone once. spaCy stores strings as integers internally, and the plain attribute
gives you the integer.

In [ ]:
token = doc[4]           # "book"

print("token.text =", token.text)
print("token.pos  =", token.pos, " (the integer spaCy stores)")
print("token.pos_ =", token.pos_, "  (the string you wanted)")
# Verified output:
#   token.text = book
#   token.pos  = 100  (the integer spaCy stores)
#   token.pos_ = VERB   (the string you wanted)

The rule has no exceptions: **an attribute with a trailing underscore is the readable string**.
`lemma` and `lemma_`, `dep` and `dep_`, `ent_type` and `ent_type_`. If a print shows you a large
integer, you left the underscore off.

### Slicing gives you a Span

Slicing a `Doc` gives a `Span` — a contiguous stretch of tokens that still knows where it came
from. Entities and noun phrases are both `Span` objects, so these attributes are worth seeing
once on a slice you made yourself.

In [ ]:
span = doc[5:8]

print("text:      ", span.text)
print("tokens:    ", [t.text for t in span])
print("characters:", span.start_char, "to", span.end_char)
print("root:      ", span.root.text)
# Verified output:
#   text:       the 9:30 flight
#   tokens:     ['the', '9:30', 'flight']
#   characters: 20 to 35
#   root:       flight

`start_char` and `end_char` are positions in the original string, which is what you need when the
answer has to be highlighted in the source text or written back into a database with a record of
where it came from. Lab 2 extracts amounts and dates from bank SMS with regular expressions and
gets the same information there from `match.start()`; this is its equivalent when the extractor
is a model rather than a pattern.

### Exercise 1

Print every token of the sentence below that is **not** a stop word and **not** punctuation,
together with its lemma and its coarse tag. This three-line filter is the preprocessing step you
will write over and over.

In [ ]:
text = "The committee has not approved the revised budgets for the new laboratories."

# YOUR CODE HERE
# for token in nlp(text):
#     skip if token.is_stop or token.is_punct
#     print token.text, token.lemma_, token.pos_

Notice what the stop list took: `has` and `not` are both on it. Dropping `not` from a sentence
about a committee reverses what the sentence says, and nothing in the code will tell you. Session
4 makes this point with a review rather than a committee; it is the same point, and it is the
reason a stop list is a decision rather than a default.

---
## Part 2 — Sentences

`doc.sents` gives the sentences. Unlike NLTK's `sent_tokenize`, which uses a model trained only
to find sentence boundaries, spaCy gets the boundaries from the dependency parser — the same
component that works out the grammatical structure.

In [ ]:
doc = nlp("Dr. Rao arrived at 9 a.m. The meeting had already started. "
          "He didn't mind; the agenda was short.")

for i, sent in enumerate(doc.sents, 1):
    print(f"  {i}. {sent.text}")
# Verified output:
#     1. Dr. Rao arrived at 9 a.m.
#     2. The meeting had already started.
#     3. He didn't mind; the agenda was short.

`Dr.` and `a.m.` did not end a sentence, and the semicolon did not either. Note also that `sents`
is a generator, not a list — `len(doc.sents)` raises a `TypeError`, and you want
`len(list(doc.sents))`. That is a two-minute confusion the first time you meet it.

---
## Part 3 — Named entities

A **named entity** is a span of text referring to a real-world thing with a name: a person, an
organisation, a place, a date, a money amount. Session 7 is the lesson; here we are learning to
read `doc.ents`.

In [ ]:
doc = nlp("The Reserve Bank of India was founded in Kolkata in 1935 "
          "and moved to Bombay two years later.")

for ent in doc.ents:
    print(f"  {ent.text:12} {ent.label_:10} chars {ent.start_char:3}-{ent.end_char:3}  "
          f"{spacy.explain(ent.label_)}")
# Verified output:
#     The Reserve Bank of India ORG        chars   0- 25  Companies, agencies, institutions, etc.
#     Kolkata      GPE        chars  41- 48  Countries, cities, states
#     1935         DATE       chars  52- 56  Absolute or relative dates or periods
#     Bombay       GPE        chars  70- 76  Countries, cities, states
#     two years later DATE       chars  77- 92  Absolute or relative dates or periods

`spacy.explain` takes any label the library uses — entity labels, POS tags, dependency labels —
and returns a description. It is the fastest way to read output you do not recognise, and it
works offline.

Entities are spans, not tokens: `The Reserve Bank of India` is five tokens and one entity, and
`two years later` is three. To go the other way and ask a token which entity it belongs to, use
the BIO scheme, where `B` marks the beginning of an entity, `I` a token inside one, and `O` a
token outside any. Session 7 explains why tagging is formulated this way.

In [ ]:
for token in doc[:12]:
    print(f"  {token.text:10} {token.ent_iob_} {token.ent_type_}")
# Verified output:
#     The        B ORG
#     Reserve    I ORG
#     Bank       I ORG
#     of         I ORG
#     India      I ORG
#     was        O
#     founded    O
#     in         O
#     Kolkata    B GPE
#     in         O
#     1935       B DATE
#     and        O

### Exercise 2

Extract, from the paragraph below, every organisation and every money amount — and only those.
Print them as two lists.

Then read the output critically. The model was trained on American news text from the 2000s, and
it will be wrong at least once here. Say which one is wrong before you look at the note below.

In [ ]:
news = ("Reliance Industries reported a quarterly profit of Rs 18,951 crore on Friday. "
        "Tata Consultancy Services, based in Mumbai, announced a dividend of $2.5 billion. "
        "Zoho, which is headquartered in Chennai, remains privately held.")

# YOUR CODE HERE
# orgs   = [ent.text for ent in nlp(news).ents if ent.label_ == "ORG"]
# money  = ...

The `Rs 18,951 crore` amount is the interesting failure. An amount written in rupees and crores
is money to any reader in India, and the model does not label the whole of it that way, because
almost nothing in its training data was written this way. Nothing in the output marks this as a
guess — a wrong label is returned exactly like a right one.

This is the honest position on off-the-shelf NER: it is good on text that resembles what it was
trained on, and it degrades quietly on text that does not. The three responses are to add rules
for the patterns you care about (part 5), to fine-tune the model on your own annotated text, or
to accept the errors and measure them. Lab 2 is the first response taken as far as it will go —
regular expressions over bank SMS, and the point at which they stop working.

---
## Part 4 — The dependency parse, and what it is good for

The parser links each token to the word it depends on, producing a tree over the sentence. We do
not teach the formalism in this course, but two things built on top of it are useful now.

`doc.noun_chunks` gives the noun phrases — a noun and the words describing it — which is often a
better unit than the single word for information extraction.

In [ ]:
doc = nlp("The new campus library issued three hundred rare books to visiting research students.")

print("noun chunks:")
for chunk in doc.noun_chunks:
    print(f"  {chunk.text:35} root: {chunk.root.text}")
# Verified output:
#   noun chunks:
#     The new campus library              root: library
#     three hundred rare books            root: books
#     research students                   root: students

The other useful piece is `token.head` and `token.children`, which walk the tree. Asking "who
did what to whom" is asking for a verb, its subject (`nsubj`) and its object (`dobj`).

In [ ]:
for token in doc:
    if token.dep_ in ("nsubj", "dobj"):
        print(f"  {token.text:10} is the {token.dep_:6} "
              f"({spacy.explain(token.dep_)}) of '{token.head.text}'")
# Verified output:
#     library    is the nsubj  (nominal subject) of 'issued'
#     books      is the dobj   (direct object) of 'issued'
#     students   is the dobj   (direct object) of 'visiting'

That is a three-line relation extractor, and its output here is two-thirds right: the library
issued books. The third row is an artefact — the parser read `visiting` as a verb with
`students` as its object, so a phrase that is really one noun chunk came out as a clause. It will
not survive a passive, a coordination or a relative clause without more care either. For a first
pass over regular prose, such as transaction descriptions or news leads, it still does real work
— the same extraction job lab 2 does with regular expressions, one level up.

---
## Part 5 — Rules, where a model is the wrong tool

Not everything needs a model. If the pattern is fixed and you can write it down, write it down:
a rule is exact, needs no training data, and never guesses. spaCy's `Matcher` matches patterns
over **tokens and their attributes**, which is what makes it more than a regular expression — it
can say "a number, then the word crore", using the tagger's decision about what is a number.

In [ ]:
from spacy.matcher import Matcher

matcher = Matcher(nlp.vocab)

# One dict per token. "LOWER" matches the lowercased text, "LIKE_NUM" matches
# anything the pipeline thinks is a number — including "eighteen" and "18,951".
matcher.add("INR_AMOUNT", [
    [{"LOWER": {"IN": ["rs", "inr", "₹"]}},          # the unit
     {"LOWER": ".", "OP": "?"},                       # "Rs." is two tokens
     {"LIKE_NUM": True},                              # the number
     {"LOWER": {"IN": ["crore", "lakh"]}, "OP": "?"}],  # an optional scale
])

doc = nlp("Reliance reported Rs 18,951 crore, up from Rs 200 crore, and paid Rs. 5 for a stamp.")

for match_id, start, end in matcher(doc):
    print("  ", doc[start:end].text)
# Verified output:
#      Rs 18,951
#      Rs 18,951 crore
#      Rs 200
#      Rs 200 crore
#      Rs. 5

`"OP": "?"` marks a token as optional, which is how one pattern covers `Rs 200 crore` and
`Rs. 5` at once. It is also why the same amount is reported twice: `Rs 18,951` matches with the
optional token absent, and `Rs 18,951 crore` matches with it present. The `Matcher` returns every
match, overlaps included, and leaves the choice to you. `filter_spans` makes the usual choice —
keep the longest span, drop anything overlapping it.

In [ ]:
from spacy.util import filter_spans

spans = [doc[start:end] for _, start, end in matcher(doc)]

for span in filter_spans(spans):
    print("  ", span.text)
# Verified output:
#      Rs 18,951 crore
#      Rs 200 crore
#      Rs. 5

Three amounts, each once — including `Rs 18,951 crore`, which the statistical model got wrong in
exercise 2. Rules and models are complementary rather than rival: use a rule where the pattern is
stable and you can enumerate it, and a model where it is not. A production system usually runs
both and prefers the rule.

---
## Part 6 — Two things about speed and vectors

**Use `nlp.pipe` for many documents.** Calling `nlp(text)` in a loop re-enters the pipeline once
per document; `nlp.pipe` batches them, which is measurably faster and is how the labs process a
corpus.

**Disable what you do not need.** If you only want entities, running the parser is wasted work.

In [ ]:
import time

reviews = ["The Bengaluru office of Infosys announced a hiring freeze in March."] * 400

start = time.perf_counter()
[nlp(r) for r in reviews]
loop = time.perf_counter() - start

start = time.perf_counter()
list(nlp.pipe(reviews, batch_size=50))
piped = time.perf_counter() - start

start = time.perf_counter()
list(nlp.pipe(reviews, batch_size=50, disable=["parser", "lemmatizer", "attribute_ruler"]))
lean = time.perf_counter() - start

print(f"  loop of nlp()          {loop:5.2f} s")
print(f"  nlp.pipe               {piped:5.2f} s")
print(f"  nlp.pipe, ner only     {lean:5.2f} s")
# Verified output:
#     loop of nlp()           0.75 s
#     nlp.pipe                0.45 s
#     nlp.pipe, ner only      0.36 s

The exact numbers depend on the machine, and on Colab they will differ from these. The ordering
will not.

**The small model has no word vectors.** This matters because `token.similarity` and
`doc.similarity` exist and will return a number anyway.

In [ ]:
import warnings

print("word-vector table:", nlp.vocab.vectors.shape)
print()

doc = nlp("king queen dosa")
king, queen, dosa = doc[0], doc[1], doc[2]

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    print("king ~ queen:", round(king.similarity(queen), 3))
    print("king ~ dosa :", round(king.similarity(dosa), 3))

print()
print("warning raised:", str(caught[0].message)[:80], "...")
# Verified output:
#   word-vector table: (0, 0)
#
#   king ~ queen: 0.579
#   king ~ dosa : 0.573
#
#   warning raised: [W007] The model you're using has no word vectors loaded, so the result of the T ...

The vector table is empty — `(0, 0)` rows by columns — because `en_core_web_sm` ships without
one. The numbers still come back, computed from the context tensors `tok2vec` produces, and they
are useless: *king* is as close to *dosa* as it is to *queen*, to three decimal places. spaCy
does warn, and this cell catches the warning and prints it, because in a notebook it is easy to
scroll past.

Do not use these numbers as word similarities. If you want vectors, use `en_core_web_md`, which
carries a real table, or train your own with Word2Vec — which is session 21 and lab 8, where
similarity is the whole point and this shortcut would quietly ruin the result.

### Exercise 3

Take the three sentences below, run them through `nlp.pipe`, and for each one print the entities
it contains and the subject of its main verb. This combines part 3 and part 4, and it is the
shape of the extraction loop in lab 2.

In [ ]:
sentences = [
    "Wipro opened a new campus in Hyderabad last April.",
    "The RBI raised the repo rate by 25 basis points.",
    "Students from Manipal won the competition in Singapore.",
]

# YOUR CODE HERE

Three sentences, and one clear error: `Manipal` came back as `WORK_OF_ART`. It is an unusual
label to see on a place name, and it is the same failure as the rupee amount in exercise 2 — a
model asked about text unlike its training data returns a confident wrong answer rather than no
answer. Checking the output on your own text, by eye, before trusting it, is not optional.

---
## What to remember

| you want | the call |
| --- | --- |
| a pipeline | `nlp = spacy.load("en_core_web_sm")` |
| everything, computed | `doc = nlp(text)` |
| the readable value | any attribute with a trailing underscore: `pos_`, `lemma_`, `dep_` |
| sentences | `list(doc.sents)` |
| entities | `doc.ents`, each with `.text`, `.label_`, `.start_char` |
| what a label means | `spacy.explain("ORG")` |
| noun phrases | `doc.noun_chunks` |
| the tree | `token.head`, `token.children`, `token.dep_` |
| a rule | `Matcher(nlp.vocab)`, one dict per token |
| many documents | `nlp.pipe(texts, batch_size=50)` |

And the three things that will actually bite you: the model is a separate download from the
library; an attribute without its trailing underscore is an integer; and `en_core_web_sm` has no
word vectors, so its `similarity()` is not a word similarity.

**Next:** primer 3 is NumPy, which is where Unit II starts — once text has become numbers, every
operation in this course is an array operation.